# 4.1 Advanced RAG: HyDE & Re-ranking

**Fix the cases where naive RAG fails.**

In this notebook you will:
- Build a baseline RAG pipeline and see where it fails
- Learn and implement **HyDE** (Hypothetical Document Embeddings)
- Learn and implement **re-ranking** with cross-encoders
- Compare naive vs advanced techniques side by side

> **Requirements:** Groq API key from [console.groq.com](https://console.groq.com)

## 1. Setup

In [ ]:
# Install all required packages:
# - cross-encoder models are part of sentence-transformers
# - we add langchain-groq for the LLM
!pip install langchain langchain-groq langchain-community chromadb sentence-transformers -q

In [ ]:
import os
from # getpass removed import # getpass removed

# Set up Groq API key for the LLM
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = # getpass removed("Enter your Groq API key: ")

print("API key is set.")

## 2. Build the Baseline RAG Pipeline

First, let's build a standard (naive) RAG pipeline — same approach as notebook 3.1. This will be our baseline for comparison.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Knowledge Base ---
# Airline policies with enough detail that naive retrieval can struggle
documents_text = [
    {
        "text": """FLIGHT CHANGE POLICY: Passengers may change their flight date or time up to 4 hours 
before departure. Economy class changes incur a $75 fee plus any fare difference. Business and 
First class changes are free of charge. Changes within 4 hours of departure are subject to 
availability and a $150 last-minute change fee. Same-day standby is available for Gold and 
Platinum loyalty members at no extra cost. If the airline cancels or delays a flight by more 
than 3 hours, passengers are entitled to free rebooking on the next available flight or a full refund.""",
        "source": "flight_change_policy"
    },
    {
        "text": """COMPENSATION FOR DELAYS AND CANCELLATIONS: For delays over 3 hours on flights within 
our network, passengers receive meal vouchers ($15 per meal period). For delays over 6 hours or 
overnight delays, hotel accommodation is provided at partner hotels. If a flight is cancelled, 
passengers can choose between rebooking on the next available flight at no cost or receiving a 
full refund to the original payment method. EU regulation passengers on flights over 3500km 
delayed more than 4 hours are entitled to EUR 600 compensation.""",
        "source": "compensation_policy"
    },
    {
        "text": """SPECIAL ASSISTANCE SERVICES: Wheelchair assistance is available at all airports at no 
charge. Passengers must request wheelchair service at least 48 hours before departure. Electric 
wheelchairs and mobility aids are transported free of charge and do not count toward baggage 
allowance. Visually impaired passengers traveling with guide dogs may board with their animal 
in the cabin at no extra fee. Hearing-impaired passengers can request visual announcements 
during boarding and inflight.""",
        "source": "special_assistance"
    },
    {
        "text": """INFANT AND CHILD TRAVEL POLICY: Infants under 2 years travel on a parent's lap at 10% 
of the adult fare. Child passengers aged 2-11 receive a 25% discount on all fare classes. 
Bassinets are available on long-haul flights for infants — request at booking. Unaccompanied 
minors (ages 5-14) can travel with our escort service for $100 per segment. Children under 5 
cannot travel unaccompanied. Car seats approved by aviation authorities can be used onboard 
if a separate seat is purchased.""",
        "source": "child_travel_policy"
    },
    {
        "text": """PET TRAVEL GUIDELINES: Small pets (under 8kg including carrier) can travel in the cabin 
for $50 per segment. Carrier must fit under the seat (max 40x30x20cm). Larger pets travel in 
the climate-controlled cargo hold for $150 per segment. All pets need a health certificate 
issued within 10 days of travel. Brachycephalic (snub-nosed) breeds are not accepted in cargo 
during summer months due to health risks. Emotional support animals are no longer accepted — 
only trained service animals fly free.""",
        "source": "pet_travel_policy"
    },
    {
        "text": """DANGEROUS GOODS AND RESTRICTED ITEMS: Lithium batteries over 100Wh must be approved 
by the airline before travel. Spare lithium batteries must be carried in hand luggage only. 
Flammable liquids, explosives, and compressed gases are strictly prohibited. E-cigarettes 
and vaping devices must be in carry-on baggage and cannot be charged onboard. Duty-free 
alcohol is permitted in carry-on if sealed in tamper-evident bags. Matches and lighters: 
one small lighter per person is allowed in carry-on but not in checked luggage.""",
        "source": "restricted_items"
    },
]

# Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_chunks = []
for raw_doc in documents_text:
    doc = Document(page_content=raw_doc["text"], metadata={"source": raw_doc["source"]})
    chunks = text_splitter.split_documents([doc])
    all_chunks.extend(chunks)

# Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="advanced_rag_baseline"
)

# Create the retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Create the LLM and prompt
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on this context. If the answer is not in the context, say so.

Context:
{context}

Question: {question}

Answer:
""")
parser = StrOutputParser()


def format_docs(docs):
    """Combine documents into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


# Build the naive RAG chain
naive_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

print(f"Baseline pipeline ready: {len(all_chunks)} chunks indexed")

## 3. Where Naive RAG Fails

Naive RAG works well for straightforward queries, but it struggles with:
- **Abstract/complex questions**: queries that don't use the same words as the documents
- **Multi-hop questions**: answers that require combining info from multiple chunks
- **Retrieval noise**: when irrelevant chunks are retrieved alongside relevant ones

Let's see some failure cases.

In [ ]:
# This query is phrased differently from any document text.
# "rights" and "consumer protection" don't appear in our docs.
hard_query = "What are my rights as a consumer if my flight gets cancelled?"

# Let's see what the retriever actually finds
naive_docs = retriever.invoke(hard_query)

print("Query:", hard_query)
print("\nRetrieved chunks (naive):")
print("=" * 60)
for i, doc in enumerate(naive_docs):
    print(f"\nChunk {i+1} [{doc.metadata['source']}]:")
    print(f"  {doc.page_content[:200]}")

# Get the naive answer
naive_answer = naive_chain.invoke(hard_query)
print("\n" + "=" * 60)
print(f"Naive RAG Answer: {naive_answer}")

## 4. HyDE: Hypothetical Document Embeddings

**The problem:** The user's query is phrased like a question, but our documents are phrased as policy statements. The embedding vectors end up far apart even when they're about the same topic.

**The solution (HyDE):**
1. Ask the LLM to **generate a hypothetical answer** (even if it's made up)
2. Embed **that answer** instead of the original query
3. Search the vector store with the hypothetical answer's embedding

Why does this work? The hypothetical answer is phrased like a **document** (not a question), so its embedding is closer to the actual documents!

```
Without HyDE:  Question embedding ──→ search (sometimes misses)
With HyDE:     Question → LLM → Hypothetical answer embedding ──→ search (better match!)
```

In [ ]:
# Step 1: Create a prompt that generates a hypothetical document.
# The LLM doesn't need to be accurate — it just needs to produce text
# that SOUNDS like our source documents.
hyde_prompt = ChatPromptTemplate.from_template("""
Write a short, factual passage that would answer this question.
Write it as if it were from an official airline policy document.
Do not say "I don't know" — write a plausible answer even if you're not sure.

Question: {question}

Passage:
""")

# Step 2: Build the HyDE chain
# This chain generates a hypothetical answer from the question
hyde_chain = hyde_prompt | llm | parser

# Step 3: Generate a hypothetical document for our hard query
hypothetical_doc = hyde_chain.invoke({"question": hard_query})

print("Original query:")
print(f"  {hard_query}")
print("\nHypothetical document (generated by LLM):")
print(f"  {hypothetical_doc}")

In [ ]:
# Step 4: Search using the hypothetical document instead of the original query.
# The hypothetical doc uses language similar to actual policy documents,
# so it matches better in the embedding space.
hyde_docs = vectorstore.similarity_search(hypothetical_doc, k=4)

print("Retrieved chunks (HyDE):")
print("=" * 60)
for i, doc in enumerate(hyde_docs):
    print(f"\nChunk {i+1} [{doc.metadata['source']}]:")
    print(f"  {doc.page_content[:200]}")

## 5. Compare: Naive vs HyDE Side by Side

In [ ]:
def rag_with_hyde(question):
    """
    Full HyDE RAG pipeline:
    1. Generate hypothetical answer
    2. Search using hypothetical answer
    3. Feed real results + original question to LLM
    """
    # Generate hypothetical document
    hypothetical = hyde_chain.invoke({"question": question})
    
    # Search with the hypothetical document embedding
    docs = vectorstore.similarity_search(hypothetical, k=4)
    
    # Format the retrieved documents as context
    context = format_docs(docs)
    
    # Generate answer using REAL documents (not the hypothetical one!)
    chain = prompt | llm | parser
    answer = chain.invoke({"context": context, "question": question})
    
    return answer, docs


# Test queries where HyDE should help
test_queries = [
    "What are my rights as a consumer if my flight gets cancelled?",
    "How does the airline accommodate travelers with special needs?",
    "What should I know about bringing batteries on a plane?",
]

for query in test_queries:
    print(f"\nQuestion: {query}")
    print("=" * 60)
    
    # Naive RAG answer
    naive_answer = naive_chain.invoke(query)
    naive_sources = set(d.metadata["source"] for d in retriever.invoke(query))
    
    # HyDE RAG answer
    hyde_answer, hyde_retrieved = rag_with_hyde(query)
    hyde_sources = set(d.metadata["source"] for d in hyde_retrieved)
    
    print(f"\nNaive RAG (sources: {naive_sources}):")
    print(f"  {naive_answer[:300]}")
    print(f"\nHyDE RAG (sources: {hyde_sources}):")
    print(f"  {hyde_answer[:300]}")
    print("-" * 60)

## 6. Re-ranking with Cross-Encoders

**The problem:** The retriever returns the top-k documents by embedding similarity, but embedding similarity is a **rough estimate**. Some irrelevant documents sneak in.

**The solution (Re-ranking):**
1. Retrieve a **larger** set (e.g., 10 documents) — cast a wide net
2. Use a **cross-encoder** to re-score each document against the query
3. Keep only the **top 3** after re-scoring — much more accurate

**Why is cross-encoder better?**
- **Bi-encoder** (what we used so far): encodes query and document **separately**, then compares. Fast but less accurate.
- **Cross-encoder**: reads query and document **together** as one input. Slower but much more accurate — it can see the relationship directly.

In [ ]:
from sentence_transformers import CrossEncoder

# CrossEncoder scores query-document pairs together (not independently).
# Slower but much more accurate than bi-encoder similarity.
# ms-marco-MiniLM-L-6-v2 is trained on search relevance data — perfect for re-ranking.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-encoder re-ranker loaded!")

In [ ]:
import numpy as np

def retrieve_and_rerank(query, vectorstore, reranker, initial_k=10, final_k=3):
    """
    Two-stage retrieval:
    Stage 1: Retrieve initial_k documents using fast bi-encoder search
    Stage 2: Re-rank using accurate cross-encoder, keep top final_k
    """
    # Stage 1: Fast retrieval — cast a wide net
    initial_docs = vectorstore.similarity_search(query, k=initial_k)
    
    # Stage 2: Re-rank — cross-encoder scores each (query, document) pair
    # We create pairs of [query, document_text] for the cross-encoder
    pairs = [[query, doc.page_content] for doc in initial_docs]
    
    # The cross-encoder returns a relevance score for each pair
    scores = reranker.predict(pairs)
    
    # Sort documents by cross-encoder score (highest first)
    scored_docs = list(zip(scores, initial_docs))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    # Return only the top final_k documents
    return scored_docs[:final_k]


# Test the re-ranker
query = "Can I bring my cat on the plane?"
reranked = retrieve_and_rerank(query, vectorstore, reranker)

print(f"Query: '{query}'")
print(f"\nTop 3 after re-ranking:")
for score, doc in reranked:
    print(f"\n  Score: {score:.4f} [{doc.metadata['source']}]")
    print(f"  {doc.page_content[:150]}...")

## 7. Compare: With and Without Re-ranking

In [ ]:
def rag_with_reranking(question):
    """
    RAG pipeline with cross-encoder re-ranking.
    Retrieve 10 → re-rank → keep top 3 → generate answer.
    """
    # Retrieve and re-rank
    reranked_results = retrieve_and_rerank(question, vectorstore, reranker)
    
    # Extract just the documents (without scores)
    docs = [doc for _, doc in reranked_results]
    
    # Format and generate
    context = format_docs(docs)
    chain = prompt | llm | parser
    answer = chain.invoke({"context": context, "question": question})
    
    return answer, reranked_results


# Test queries
comparison_queries = [
    "Can I bring my cat on the plane?",
    "What happens if my flight is delayed overnight?",
    "Can a 7-year-old child fly alone?",
]

for query in comparison_queries:
    print(f"\nQuestion: {query}")
    print("=" * 60)
    
    # Without re-ranking (naive)
    naive_answer = naive_chain.invoke(query)
    naive_docs = retriever.invoke(query)
    
    # With re-ranking
    reranked_answer, reranked_results = rag_with_reranking(query)
    
    print("\nWithout re-ranking:")
    print(f"  Sources: {[d.metadata['source'] for d in naive_docs]}")
    print(f"  Answer: {naive_answer[:250]}")
    
    print("\nWith re-ranking:")
    print(f"  Sources: {[d.metadata['source'] for _, d in reranked_results]}")
    print(f"  Scores: {[f'{s:.3f}' for s, _ in reranked_results]}")
    print(f"  Answer: {reranked_answer[:250]}")
    print("-" * 60)

## 8. Combining HyDE + Re-ranking

For the best results, we can combine both techniques:
1. **HyDE** to improve initial retrieval
2. **Re-ranking** to filter the results

In [ ]:
def advanced_rag(question):
    """
    Full advanced RAG pipeline combining HyDE and re-ranking.
    
    Flow:
    1. Generate hypothetical document (HyDE)
    2. Retrieve 10 documents using HyDE embedding
    3. Re-rank with cross-encoder, keep top 3
    4. Generate answer with LLM
    """
    # Step 1: Generate hypothetical document
    hypothetical = hyde_chain.invoke({"question": question})
    
    # Step 2: Retrieve using hypothetical document embedding
    initial_docs = vectorstore.similarity_search(hypothetical, k=10)
    
    # Step 3: Re-rank with cross-encoder using ORIGINAL question
    # Important: we re-rank against the original question, not the hypothetical!
    pairs = [[question, doc.page_content] for doc in initial_docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(scores, initial_docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in scored_docs[:3]]
    
    # Step 4: Generate answer
    context = format_docs(top_docs)
    chain = prompt | llm | parser
    answer = chain.invoke({"context": context, "question": question})
    
    return answer, top_docs


# Final comparison: Naive vs Advanced
final_query = "What options do I have if the airline cancels my long-distance flight?"

print(f"Question: {final_query}")
print("\n" + "=" * 60)

# Naive
naive_result = naive_chain.invoke(final_query)
print(f"NAIVE RAG:\n  {naive_result}")

# Advanced (HyDE + Re-ranking)
advanced_result, advanced_docs = advanced_rag(final_query)
print(f"\nADVANCED RAG (HyDE + Re-ranking):\n  {advanced_result}")
print(f"\n  Sources used: {[d.metadata['source'] for d in advanced_docs]}")

## 9. YOUR TURN: Test and Compare

Try your own queries! Which technique works best for different types of questions?

Ideas to try:
- Very specific factual questions ("What is the weight limit for...")
- Abstract questions ("What should I know before traveling with a disability?")
- Multi-topic questions ("Can I bring my dog and my guitar on the same flight?")

In [ ]:
# YOUR TURN: Replace this query with your own!
my_query = "Replace this with your question"

# Compare all three approaches
print(f"Question: {my_query}")
print("\n" + "=" * 60)

# 1. Naive RAG
print("\n1. NAIVE RAG:")
print(f"   {naive_chain.invoke(my_query)}")

# 2. HyDE RAG
print("\n2. HyDE RAG:")
hyde_answer, _ = rag_with_hyde(my_query)
print(f"   {hyde_answer}")

# 3. Re-ranking RAG
print("\n3. RE-RANKING RAG:")
rerank_answer, _ = rag_with_reranking(my_query)
print(f"   {rerank_answer}")

# 4. HyDE + Re-ranking
print("\n4. ADVANCED RAG (HyDE + Re-ranking):")
advanced_answer, _ = advanced_rag(my_query)
print(f"   {advanced_answer}")

## Key Takeaways

1. **Naive RAG fails** when queries are phrased differently from documents or when irrelevant chunks are retrieved
2. **HyDE** bridges the gap between question-style queries and document-style text by generating a hypothetical answer first
3. **Cross-encoder re-ranking** improves precision by scoring query-document pairs together (not independently)
4. **Combining both** gives the best results: HyDE for better recall, re-ranking for better precision

**Next up:** In notebook **5.1**, we'll learn how to **evaluate** our RAG pipeline with metrics so we can objectively measure improvements!